# Notebook 05: Feature Selection and Model Optimization

## 1. Context and Objective
In Notebook 04, we expanded our feature space to 60 predictive variables derived from the complete ENAMED student perception questionnaire (Arquivo 3). While this multivariable approach successfully resolved our local boundary issue by correctly classifying the Juiz de Fora campus as *High Performance*, we observed a slight drop in global accuracy (to 68.57%) due to the "Complexity Paradox"—noise and data redundancy among intermediate, overlapping responses.

The objective of this notebook is to optimize our predictive system. We will:
1. **Apply Feature Selection:** Discard lower-ranked perception variables based on the Feature Importance threshold from Notebook 04 to eliminate noise.
2. **Execute Hyperparameter Tuning:** Leverage `GridSearchCV` to find the optimal structure for our Random Forest model (tuning tree depth and estimators).
3. **Implement K-Fold Cross-Validation:** Replace static train/test splits with robust cross-validation to guarantee structural stability and eliminate sampling bias.

In [20]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

RAW_PATH = '../data/raw/'
PROCESSED_PATH = '../data/processed/'

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [10, 6]

df_base = pd.read_csv(os.path.join(PROCESSED_PATH, 'enamed_2025_aggregated_performance.csv'))
median_score = df_base['avg_score_general'].median()
df_base['performance_class'] = np.where(df_base['avg_score_general'] >= median_score, 1, 0)

df_arq3 = pd.read_csv(os.path.join(RAW_PATH, 'microdados_enade_2025_arq3.txt'), sep=';', decimal='.')
df_arq3_present = df_arq3[df_arq3['TP_PR_GER'] == 555].copy()

perception_cols = [col for col in df_arq3_present.columns if 'CO_RS_' in col or 'PERCEPCAO' in col]
df_features_all = pd.DataFrame({'CO_CURSO': df_base['CO_CURSO']})

for col in perception_cols:
    ct = pd.crosstab(df_arq3_present['CO_CURSO'], df_arq3_present[col], normalize='index') * 100
    ct.columns = [f'pct_{col}_{resp}' for resp in ct.columns]
    df_features_all = pd.merge(df_features_all, ct.reset_index(), on='CO_CURSO', how='left')

df_ml_multi = pd.merge(df_base, df_features_all, on='CO_CURSO', how='inner')
df_ml_multi = df_ml_multi.fillna(df_ml_multi.median(numeric_only=True))

print(f"✅ Baseline data loaded. Total features before selection: {len([c for c in df_ml_multi.columns if 'pct_' in c])}")

✅ Baseline data loaded. Total features before selection: 60


## 2. Automated Feature Selection based on Importance Threshold
We isolate the original 60 features. To clean up the dataset and resolve the complexity paradox, we will apply an automated selection process keeping only the top 20 most expressive features derived from our Random Forest importance distribution, dropping the bottom 40 columns that act as mathematical noise.

In [21]:
feature_cols_all = [col for col in df_ml_multi.columns if 'pct_' in col]
X_all = df_ml_multi[feature_cols_all]
y = df_ml_multi['performance_class']

rf_selector = RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5)
rf_selector.fit(X_all, y)

df_imp_mapping = pd.DataFrame({
    'Feature': feature_cols_all,
    'Importance': rf_selector.feature_importances_
}).sort_values(by='Importance', ascending=False)

selected_features = df_imp_mapping.head(20)['Feature'].tolist()
X_selected = df_ml_multi[selected_features]

print("=== FEATURE SELECTION COMPLETED ===")
print(f"Features kept for optimization: {len(selected_features)}")
print(f"Noise features dropped: {len(feature_cols_all) - len(selected_features)}")
print("\nTop 20 Selected Pillars:")
for f in selected_features[:20]:
    print(f"  - {f}")

=== FEATURE SELECTION COMPLETED ===
Features kept for optimization: 20
Noise features dropped: 40

Top 20 Selected Pillars:
  - pct_CO_RS_I1_C
  - pct_CO_RS_I1_D
  - pct_CO_RS_I1_B
  - pct_CO_RS_I7_D
  - pct_CO_RS_I7_B
  - pct_CO_RS_I9_B
  - pct_CO_RS_I5_D
  - pct_CO_RS_I4_B
  - pct_CO_RS_I6_A
  - pct_CO_RS_I7_C
  - pct_CO_RS_I9_A
  - pct_CO_RS_I1_.
  - pct_CO_RS_I6_B
  - pct_CO_RS_I7_A
  - pct_CO_RS_I3_C
  - pct_CO_RS_I7_E
  - pct_CO_RS_I2_.
  - pct_CO_RS_I5_C
  - pct_CO_RS_I3_.
  - pct_CO_RS_I4_D


## 3. Hyperparameter Tuning via GridSearchCV and Cross-Validation
With our filtered dataset of 20 features, we implement a `GridSearchCV` to test combinations of `n_estimators` and `max_depth`. Instead of a static split, we integrate a 5-Fold Cross-Validation strategy to guarantee that our performance metrics are stable across different data subsets and free from split biases.

In [22]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

param_grid = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 4, 5, 6, 7],
    'criterion': ['gini', 'entropy']
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1
)

print("Optimizing Random Forest hyperparameters over the 20-feature space...")
grid_search.fit(X_selected, y)

print("\n=== TUNING RESULTS ===")
print(f"Best Hyperparameters Found: {grid_search.best_params_}")
print(f"Best Cross-Validated Accuracy: {grid_search.best_score_:.4f}")

best_rf = grid_search.best_estimator_

Optimizing Random Forest hyperparameters over the 20-feature space...

=== TUNING RESULTS ===
Best Hyperparameters Found: {'criterion': 'entropy', 'max_depth': 6, 'n_estimators': 50}
Best Cross-Validated Accuracy: 0.7800


## 4. Final Evaluation and Local UFJF Prediction Confidence
We evaluate the optimized model's global classification footprint and rerun our regional validation for the UFJF campuses to inspect if the feature selection and hyperparameter tuning raised our prediction confidence.

In [25]:
scaler_final = StandardScaler()
X_selected_scaled = scaler_final.fit_transform(X_selected)

best_rf.fit(X_selected_scaled, y)

df_ufjf = df_ml_multi[df_ml_multi['CO_CURSO'].isin([13103, 5001167])].copy()

if not df_ufjf.empty:
    X_ufjf_local = df_ufjf[selected_features]
    X_ufjf_local_scaled = scaler_final.transform(X_ufjf_local)
    
    preds_ufjf = best_rf.predict(X_ufjf_local_scaled)
    probs_ufjf = best_rf.predict_proba(X_ufjf_local_scaled)
    
    print("=== OPTIMIZED UFJF REGIONAL PREDICTIONS ===")
    for idx, row in df_ufjf.reset_index(drop=True).iterrows():
        campus_name = "Juiz de Fora" if row['CO_CURSO'] == 13103 else "Governador Valadares"
        pred_class = "High Performance" if preds_ufjf[idx] == 1 else "Low Performance"
        actual_class = "High Performance" if row['performance_class'] == 1 else "Low Performance"
        confidence = probs_ufjf[idx][preds_ufjf[idx]] * 100
        
        print(f"Campus: {campus_name} ({int(row['CO_CURSO'])})")
        print(f"  Prediction: {pred_class} ({confidence:.1f}% confidence)")
        print(f"  Actual Exam Class: {actual_class}\n")
else:
    print("⚠️ UFJF courses not found in the processed subset.")

    

=== OPTIMIZED UFJF REGIONAL PREDICTIONS ===
Campus: Juiz de Fora (13103)
  Prediction: High Performance (57.6% confidence)
  Actual Exam Class: High Performance

Campus: Governador Valadares (5001167)
  Prediction: High Performance (94.6% confidence)
  Actual Exam Class: High Performance

